# BW Model Walkthrough

Step-by-step integration of the Balloon-Windkessel ODE (Friston 2003).

In [ ]:
import numpy as np
import sys; sys.path.insert(0, "..")
from src.bw_model import BWParams, simulate, bold_signal
from src.neural_generator import generate_event_related
from src.hrf import double_gamma_hrf, convolve_hrf
params = BWParams()
print(params)

In [ ]:
dt, T = 0.001, 30.0
neural = generate_event_related(3, 8.0, 1.0, T, dt, noise_std=0.005)
t = np.arange(0, T, dt)
result = simulate(neural, dt, T, params)
print("BOLD shape:", result.bold.shape, "max:", result.bold.max())

In [ ]:
from src.param_sweep import sweep_single_param, extract_bold_features
kappa_values = [0.3, 0.65, 1.0, 1.2]
kappa_sweep = sweep_single_param("kappa", kappa_values, params, neural, dt, T)
print("Sweep entries:", len(kappa_sweep))

In [ ]:
from src.delay_inject import inject_delays_bw
from src.neural_generator import generate_coupled_oscillators
T2, dt2 = 60.0, 0.001
C = np.array([[0, 0.2], [0.2, 0]])
neural_2r = generate_coupled_oscillators(2, T2, dt2, C, np.array([0.05, 0.05]))
delays = np.array([0.0, 2.0])
bold_2r = inject_delays_bw(neural_2r, delays, params, dt2, T2)
print("Delayed BOLD shape:", bold_2r.shape)

In [ ]:
from src.fc_from_bold import fc_legacy, fc_delay_corrected, fc_difference
from src.bw_model import downsample
tr = 2.0
bold_ds = np.stack([downsample(bold_2r[i], dt2, tr) for i in range(2)])
fc_leg = fc_legacy(bold_ds, tr)
fc_corr = fc_delay_corrected(bold_ds, delays, tr)
delta = fc_difference(fc_leg, fc_corr)
print("FC legacy:", fc_leg[0,1], "FC corrected:", fc_corr[0,1])

In [ ]:
from src.bifurcation import g_sweep, compute_model_fit, find_g_optimal
n_regions = 10
dist = np.abs(np.subtract.outer(np.arange(n_regions), np.arange(n_regions)))
C_10 = np.exp(-dist / 3.0); np.fill_diagonal(C_10, 0)
fc_emp = np.eye(n_regions) + 0.1 * np.random.randn(n_regions, n_regions)
fc_emp = (fc_emp + fc_emp.T) / 2; np.fill_diagonal(fc_emp, 1.0)
G_values = np.logspace(-2, 1, 5)
fc_dict = g_sweep(G_values, C_10, n_regions, T=30.0, dt=0.001, params=params, tr=2.0)
fit = compute_model_fit(G_values, fc_dict, fc_emp)
print("Optimal G:", find_g_optimal(fit))

## Summary

BW model walkthrough complete. All key components demonstrated.

In [ ]:
print("Walkthrough complete.")